<h1 style="color:#1E54B7;" align="center">CLIP Similar & Duplicate Image Finder</h1>

In [21]:
#----------------------------CLIP model + processor for image embeddings, plus standard ML/utility libraries-------------------------------------

!pip install transformers torch torchvision pillow requests --quiet

from transformers import CLIPProcessor, CLIPModel
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import os
import glob

In [22]:
#------------------------------- Folder containing the personal image dataset to search through------------------------------------
image_folder = r"D:\Images"
image_paths = glob.glob(os.path.join(image_folder, "*.*"))

print(f"Number of images found: {len(image_paths)}")

Number of images found: 142


In [23]:
#---------------------------- Using the base CLIP model (openai/clip-vit-base-patch32): good accuracy/speed tradeoff for CPU usage----------------------

model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)

images = []
valid_paths = []

for path in image_paths:
    try:
        img = Image.open(path).convert("RGB")
        images.append(img)
        valid_paths.append(path)
    except Exception as e:
        # -----------------------------Skip corrupted or unreadable files instead of stopping the whole pipeline-------
        print(f"Failed to open {path}: {e}")

print(f"Number of valid images: {len(images)}")

inputs = processor(images=images, return_tensors="pt")

with torch.no_grad():  # ----------------------------inference only, no need to track gradients------------------------------
    image_features_output = model.get_image_features(**inputs)
    #-------------------- Different transformers versions return the embedding under different attribute names
    if hasattr(image_features_output, "pooler_output"):
        image_embeddings = image_features_output.pooler_output
    elif hasattr(image_features_output, "image_embeds"):
        image_embeddings = image_features_output.image_embeds
    else:
        image_embeddings = image_features_output

print("Embedding shape for all images:", image_embeddings.shape)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of valid images: 142
Embedding shape for all images: torch.Size([142, 512])


In [24]:
# -----------------------------Pairwise cosine similarity between every image embedding (N x N matrix)---------------------------------------------

similarity_matrix = F.cosine_similarity(
    image_embeddings[:, None, :],
    image_embeddings[None, :, :],
    dim=2
)

# -----------------------------------Images scoring this high are treated as exact duplicates rather than just "similar",----------------------------
# -----------------------------------since even closely related images rarely reach this score with CLIP---------------------------------------------

DUPLICATE_THRESHOLD = 0.98


def find_similar_images(query_index, threshold=0.7, max_results=10):
    """
    Return dataset images similar to the image at `query_index`.
    Only images with similarity >= threshold are included, sorted by score.
    Each result is labeled "Exact Duplicate" or "Similar".
    """
    scores = similarity_matrix[query_index]
    scores_sorted = torch.argsort(scores, descending=True)

    results = []
    for idx in scores_sorted:
        idx = idx.item()
        if idx == query_index:
            continue
        score = scores[idx].item()
        if score < threshold:
            break  # ----------------------------------remaining scores are even lower, no need to keep checking---------------------------
        label = "Exact Duplicate" if score >= DUPLICATE_THRESHOLD else "Similar"
        results.append((valid_paths[idx], score, label))
        if len(results) == max_results:
            break
    return results

In [25]:
def show_similar_images(query_index, threshold=0.7):
    """Display the query image alongside its similar/duplicate matches."""
    similar = find_similar_images(query_index, threshold=threshold)
    n_results = len(similar)

    if n_results == 0:
        print("No similar images found above this threshold.")
        return

    fig, axes = plt.subplots(1, n_results + 1, figsize=(3 * (n_results + 1), 4))

    query_img = Image.open(valid_paths[query_index]).convert("RGB")
    axes[0].imshow(query_img)
    axes[0].set_title("Query")
    axes[0].axis("off")

    for i, (path, score, label) in enumerate(similar):
        img = Image.open(path).convert("RGB")
        axes[i + 1].imshow(img)
        # Duplicates are highlighted in red so they stand out at a glance
        title_color = "red" if label == "Exact Duplicate" else "black"
        axes[i + 1].set_title(f"{score:.2f}\n{label}", color=title_color, fontsize=9)
        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.show()

In [26]:
!pip install gradio --quiet

import gradio as gr


def build_custom_dataset(files):
    """Encode user-uploaded images into a custom dataset, stored in gr.State."""
    if not files:
        return None, None, "No files uploaded yet."

    images, paths = [], []
    for f in files:
        path = f.name if hasattr(f, "name") else f
        try:
            images.append(Image.open(path).convert("RGB"))
            paths.append(path)
        except Exception as e:
            print(f"Failed to open {path}: {e}")

    if not images:
        return None, None, "Could not read any of the uploaded files as images."

    inputs = processor(images=images, return_tensors="pt")
    with torch.no_grad():
        output = model.get_image_features(**inputs)
        if hasattr(output, "pooler_output"):
            embeddings = output.pooler_output
        elif hasattr(output, "image_embeds"):
            embeddings = output.image_embeds
        else:
            embeddings = output

    status = f"Custom dataset ready: {len(images)} image(s) indexed."
    return embeddings, paths, status


def clear_custom_dataset():
    """Discard the custom dataset and fall back to the notebook's own dataset."""
    return None, None, "Custom dataset cleared — using the notebook dataset again."


def gradio_find_similar(query_image, threshold, custom_embeddings, custom_paths):
    """
    Find similar images using either the user's custom dataset (if built)
    or the dataset already loaded earlier in this notebook (image_embeddings, valid_paths).
    """
    if custom_embeddings is not None and custom_paths:
        embeddings, paths, source = custom_embeddings, custom_paths, "your uploaded dataset"
    else:
        embeddings, paths, source = image_embeddings, valid_paths, "this notebook's dataset"

    query_input = processor(images=[query_image], return_tensors="pt")
    with torch.no_grad():
        query_output = model.get_image_features(**query_input)
        if hasattr(query_output, "pooler_output"):
            query_embedding = query_output.pooler_output
        elif hasattr(query_output, "image_embeds"):
            query_embedding = query_output.image_embeds
        else:
            query_embedding = query_output

    scores = F.cosine_similarity(query_embedding, embeddings, dim=1)
    scores_sorted = torch.argsort(scores, descending=True)

    result_images, result_captions = [], []
    duplicate_count = 0

    for idx in scores_sorted:
        idx = idx.item()
        score = scores[idx].item()
        if score < threshold:
            break
        img = Image.open(paths[idx]).convert("RGB")
        if score >= DUPLICATE_THRESHOLD:
            caption = f"Exact Duplicate ({score:.2f})"
            duplicate_count += 1
        else:
            caption = f"Similar ({score:.2f})"
        result_images.append(img)
        result_captions.append(caption)
        if len(result_images) == 10:
            break

    if not result_images:
        return [], f"No similar images found in {source}. Try lowering the threshold."

    note = f"Searched {source} — found {len(result_images)} match(es), {duplicate_count} exact duplicate(s)."
    return list(zip(result_images, result_captions)), note


custom_theme = gr.themes.Soft(primary_hue="blue", secondary_hue="blue", neutral_hue="slate").set(
    body_background_fill="#F4F8FF",
    block_background_fill="#FFFFFF",
    block_border_color="#DCE8FF",
    button_primary_background_fill="#1E54B7",
    button_primary_background_fill_hover="#3B82F6",
    button_primary_text_color="#FFFFFF",
)

custom_css = """
#title-block {text-align:center; padding: 6px 0 0 0;}
#title-block h1 {color:#0F2A5C; font-weight:800;}
#subtitle {text-align:center; color:#0F2A5C; margin-top:-8px; font-weight:600;}
.gradio-container {max-width: 1100px !important; margin: auto;}

#dataset-note {
    color:#0F2A5C;
    font-weight:600;
    background:#EEF4FF;
    padding:10px 14px;
    border-radius:8px;
    margin-bottom:8px;
}

/* Similarity threshold slider tooltip */
#threshold-slider { position: relative; }
#threshold-slider::after {
    content: "Lower = looser matches (more results) · Higher = stricter matches (fewer, closer results)";
    display: none;
    position: absolute; top: -34px; left: 0;
    background: #0F2A5C; color: #fff; font-size: 12px; padding: 6px 10px;
    border-radius: 6px; white-space: nowrap; z-index: 10;
}
#threshold-slider:hover::after { display: block; }

#build-dataset-btn, #clear-dataset-btn { position: relative; }

#build-dataset-btn::after {
    content: "Encode your uploaded images and search against them";
    display: none;
    position: absolute; top: 108%; left: 50%; transform: translateX(-50%);
    background: #0F2A5C; color: #fff; font-size: 12px; padding: 6px 10px;
    border-radius: 6px; white-space: nowrap; z-index: 10;
}
#build-dataset-btn:hover::after { display: block; }

#clear-dataset-btn::after {
    content: "Discard your custom dataset and use the notebook dataset again";
    display: none;
    position: absolute; top: 108%; left: 50%; transform: translateX(-50%);
    background: #0F2A5C; color: #fff; font-size: 12px; padding: 6px 10px;
    border-radius: 6px; white-space: nowrap; z-index: 10;
}
#clear-dataset-btn:hover::after { display: block; }

/* Accordion label + arrow styling */
#dataset-accordion .label-wrap span {
    color:#0F2A5C !important;
    font-weight:700 !important;
    font-size: 15px !important;
}
#dataset-accordion .label-wrap svg {
    width: 20px !important;
    height: 20px !important;
    color: #0F2A5C !important;
    fill: #0F2A5C !important;
    stroke: #0F2A5C !important;
}
"""

with gr.Blocks(theme=custom_theme, css=custom_css) as demo:
    custom_embeddings_state = gr.State(None)
    custom_paths_state = gr.State(None)

    with gr.Column(elem_id="title-block"):
        gr.Markdown("# 🔎 CLIP Similar & Duplicate Image Finder")
        gr.Markdown(
            "Upload an image to find visually/semantically similar images, "
            "with automatic flagging of exact duplicates.",
            elem_id="subtitle",
        )

    with gr.Row():
        with gr.Column():
            query_input = gr.Image(type="pil", label="Upload a query image")
            threshold_input = gr.Slider(
                minimum=0.5, maximum=0.95, value=0.7, step=0.01,
                label="Similarity threshold",
                elem_id="threshold-slider",
            )
            submit_btn = gr.Button("Find Similar Images", variant="primary")
        with gr.Column():
            gallery_output = gr.Gallery(label="Most similar images", columns=3, height=400)
            summary_output = gr.Textbox(label="Result summary")

    with gr.Accordion("📁 Use your own dataset (optional)", open=False, elem_id="dataset-accordion"):
        gr.Markdown(
            "By default, searches run against this notebook's dataset. "
            "Upload your own images below to search against them instead.",
            elem_id="dataset-note",
        )
        custom_files_input = gr.File(label="Upload your own images", file_count="multiple", file_types=["image"])
        with gr.Row():
            build_dataset_btn = gr.Button("Build my dataset", variant="primary", elem_id="build-dataset-btn")
            clear_dataset_btn = gr.Button("Use notebook dataset instead", variant="primary", elem_id="clear-dataset-btn")
        dataset_status = gr.Textbox(label="Dataset status", interactive=False)

    submit_btn.click(fn=gradio_find_similar,
                      inputs=[query_input, threshold_input, custom_embeddings_state, custom_paths_state],
                      outputs=[gallery_output, summary_output])

    build_dataset_btn.click(fn=build_custom_dataset, inputs=[custom_files_input],
                             outputs=[custom_embeddings_state, custom_paths_state, dataset_status])

    clear_dataset_btn.click(fn=clear_custom_dataset, inputs=[],
                             outputs=[custom_embeddings_state, custom_paths_state, dataset_status])

demo.launch()

C:\Users\Sun Store\AppData\Local\Temp\ipykernel_16732\3795658300.py:161: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css) as demo:


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
